# START HERE — TEAMMATE INSTRUCTIONS

> **Project:** Godrej Warehouse Video Intelligence  
> **Task:** YOLO11s Object Detection (5-Epoch Baseline Training)  
> **Dataset:** `MASTER_PUBLIC_V1` (72,329 images, 144,659 total files)  

Welcome! You do **NOT** need any local repository files, Python virtual environments, or Roboflow API keys to run this notebook. Everything is self-contained in this Kaggle environment.

---

### 7-Step Quick Start Checklist:
1. **Open Notebook in Kaggle:** Ensure you are viewing this notebook in edit/run mode.
2. **Enable GPU Accelerator:** Look at the right-hand panel → **Session options** → **Accelerator** → Select **GPU T4** or **GPU P100** (or **GPU T4 x2**).
3. **Turn Internet ON:** Right panel → **Session options** → **Internet** → Toggle to **On** *(Required for Ultralytics to fetch base `yolo11s.pt` weights)*.
4. **Attach Dataset:** Right panel → Click **+ Add Input** → Go to **Your Datasets** → Attach **Godrej Warehouse Master V1** (`MASTER_PUBLIC_V1`).
5. **Run Cells Sequentially:** Click **Run All** (or run Cells 1 through 8 from top to bottom).
6. **Wait for 5 Epochs to Complete:** The training will take approximately 30–40 minutes on Kaggle GPU.
7. **Download Artifacts:** When Cell 8 finishes, open the right panel → **Data** → **Output** (`/kaggle/working`) → Download `yolo11s_baseline_artifacts.zip`.

---

### Master Target Classes (7 Categories):
| Class ID | Class Name | Description |
|---|---|---|
| `0` | **person** | Warehouse workers and operators |
| `1` | **carton** | Standard cardboard boxes and packages |
| `2` | **pallet** | Wooden and plastic pallets |
| `3` | **pallet_jack** | Manual and electric pallet jacks |
| `4` | **forklift** | Warehouse forklifts and reach trucks |
| `5` | **trolley** | Hand trolleys, rolling carts |
| `6` | **truck** | Logistics trucks and delivery vehicles |


In [ ]:
# CELL 1: Environment & GPU Verification
import sys
import torch

print('=' * 70)
print('CELL 1: ENVIRONMENT & GPU VERIFICATION')
print('=' * 70)
print(f'Python Version:      {sys.version.split()[0]}')
print(f'PyTorch Version:     {torch.__version__}')
print(f'CUDA Available:      {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f'Detected GPU Count:  {gpu_count}')
    for i in range(gpu_count):
        device_name = torch.cuda.get_device_name(i)
        vram_gb = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f'  [Device {i}] {device_name} — Total VRAM: {vram_gb:.2f} GB')
    print('\nTarget Baseline Device: cuda:0 (clean, deterministic single-GPU execution)')
else:
    raise RuntimeError(
        'CUDA is NOT available!\n'
        'Please enable GPU in the right sidebar: Session options -> Accelerator -> GPU T4 or P100.'
    )


In [ ]:
# CELL 2: Install / Verify Required Packages
print('=' * 70)
print('CELL 2: VERIFY AND INSTALL PACKAGES')
print('=' * 70)

!pip install -q ultralytics pyyaml pandas matplotlib

import ultralytics
import yaml
import pandas as pd
import matplotlib

print(f'Ultralytics version: {ultralytics.__version__}')
print(f'PyYAML version:      {yaml.__version__}')
print(f'Pandas version:      {pd.__version__}')
print(f'Matplotlib version:  {matplotlib.__version__}')
print('Package verification successful.')


In [ ]:
# CELL 3: Dataset Discovery, Structure & Format Validation, and Dynamic data.yaml
from pathlib import Path
import random
import yaml

print('=' * 70)
print('CELL 3: DATASET DISCOVERY & COMPREHENSIVE PRE-TRAINING VALIDATION')
print('=' * 70)

EXPECTED_CLASSES = {
    0: 'person',
    1: 'carton',
    2: 'pallet',
    3: 'pallet_jack',
    4: 'forklift',
    5: 'trolley',
    6: 'truck',
}

EXPECTED_COUNTS = {
    'train': (57908, 57908),
    'val': (9963, 9963),
    'test': (4458, 4458),
}

# 1. Locate dataset root under /kaggle/input (fast shallow search)
input_dir = Path('/kaggle/input')
dataset_root = None

if not input_dir.exists():
    raise FileNotFoundError('/kaggle/input directory not found. This notebook must be run inside Kaggle.')

# Check top-level and 1-level deep candidates
for p in list(input_dir.iterdir()):
    if (p / 'images' / 'train').is_dir():
        dataset_root = p
        break
    if (p / 'MASTER_PUBLIC_V1' / 'images' / 'train').is_dir():
        dataset_root = p / 'MASTER_PUBLIC_V1'
        break
    for sub in [s for s in p.iterdir() if s.is_dir()]:
        if (sub / 'images' / 'train').is_dir():
            dataset_root = sub
            break
    if dataset_root:
        break

if dataset_root is None:
    available = [p.name for p in input_dir.glob('*')]
    raise FileNotFoundError(
        f'Could not locate MASTER_PUBLIC_V1 under /kaggle/input!\n'
        f'Found items in /kaggle/input: {available}\n'
        'Please attach the MASTER_PUBLIC_V1 dataset via: "+ Add Input" in the right sidebar.'
    )

print(f'Located dataset root at: {dataset_root}')

# 2. Pre-training split existence and count validation
print('\nVerifying dataset splits & file counts:')
print('-' * 70)
print(f"{'Split':<8} {'Images Found':<14} {'Expected':<12} {'Labels Found':<14} {'Expected':<12} {'Status'}")
print('-' * 70)

all_label_files = []
for split, (exp_imgs, exp_lbls) in EXPECTED_COUNTS.items():
    img_dir = dataset_root / 'images' / split
    lbl_dir = dataset_root / 'labels' / split
    
    if not img_dir.is_dir():
        raise FileNotFoundError(f'Missing images directory for split: {img_dir}')
    if not lbl_dir.is_dir():
        raise FileNotFoundError(f'Missing labels directory for split: {lbl_dir}')
        
    imgs = [f for f in img_dir.iterdir() if f.is_file()]
    lbls = [f for f in lbl_dir.iterdir() if f.is_file() and f.suffix.lower() == '.txt']
    
    is_valid = (len(imgs) == exp_imgs) and (len(lbls) == exp_lbls)
    status = 'PASS' if is_valid else 'FAIL'
    print(f"{split:<8} {len(imgs):<14,d} {exp_imgs:<12,d} {len(lbls):<14,d} {exp_lbls:<12,d} {status}")
    
    assert len(imgs) == exp_imgs, f'Image count mismatch in {split}: got {len(imgs)}, expected {exp_imgs}'
    assert len(lbls) == exp_lbls, f'Label count mismatch in {split}: got {len(lbls)}, expected {exp_lbls}'
    all_label_files.extend(lbls)

print('-' * 70)
print('Dataset count verification PASSED!')

# 3. Label syntax and coordinate validity audit (sample 1,000 label files)
print('\nPerforming sample label coordinate & class ID audit (1,000 files)...')
sample_labels = random.sample(all_label_files, min(1000, len(all_label_files)))
verified_boxes = 0

for lbl_file in sample_labels:
    with open(lbl_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, start=1):
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) != 5:
                raise ValueError(f'Malformed label format in {lbl_file.name}:{line_num} — expected 5 values, got {len(parts)}')
            cls_id = int(parts[0])
            x, y, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            
            if cls_id not in EXPECTED_CLASSES:
                raise ValueError(f'Illegal class ID {cls_id} in {lbl_file.name}:{line_num}. Valid IDs: 0-6')
            if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                raise ValueError(f'Coordinates out of bounds in {lbl_file.name}:{line_num} — (x={x}, y={y}, w={w}, h={h})')
            verified_boxes += 1

print(f'Label syntax check PASSED: {verified_boxes:,d} boxes verified across sample.')

# 4. Generate dynamic working YAML in /kaggle/working/data_kaggle.yaml
data_yaml_content = {
    'path': str(dataset_root.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': EXPECTED_CLASSES,
}

kaggle_yaml_path = Path('/kaggle/working/data_kaggle.yaml')
with open(kaggle_yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_yaml_content, f, sort_keys=False)

print(f'\nGenerated working YAML: {kaggle_yaml_path}')
print('=' * 40)
print(kaggle_yaml_path.read_text(encoding='utf-8'))
print('=' * 40)


In [ ]:
# CELL 4: Load YOLO11s Pretrained Model
from ultralytics import YOLO

print('=' * 70)
print('CELL 4: LOAD YOLO11s MODEL')
print('=' * 70)

# Ultralytics downloads official pretrained weights yolo11s.pt automatically from GitHub releases
model = YOLO('yolo11s.pt')

print('\nModel loaded successfully:')
model.info()


In [ ]:
# CELL 5: Run 5-Epoch Baseline Training
from pathlib import Path

print('=' * 70)
print('CELL 5: 5-EPOCH BASELINE TRAINING')
print('=' * 70)

# Baseline Parameter Rationale:
# - batch=32: Optimal throughput and stability on 16GB Kaggle GPUs (T4/P100), avoiding AutoBatch profiling delays/OOM spikes.
# - device=0: Clean single-GPU baseline execution.
# - workers=4: Fully utilizes Kaggle's 4 vCPUs without IPC dataloader bottlenecks.
# - amp=True: FP16 mixed precision for ~2.5x speedup on Tensor Cores.
# - cache=False: Prevents filling Kaggle's 30GB RAM with 57k images.
# - seed=42: Ensures reproducible training behavior.

train_results = model.train(
    data='/kaggle/working/data_kaggle.yaml',
    epochs=5,
    imgsz=640,
    batch=32,
    device=0,
    workers=4,
    project='/kaggle/working/runs',
    name='yolo11s_baseline',
    exist_ok=True,
    pretrained=True,
    amp=True,
    cache=False,
    seed=42,
    save=True,
    plots=True,
)

print('\n' + '=' * 70)
print('5-EPOCH BASELINE TRAINING FINISHED SUCCESSFULLY')
print('=' * 70)


In [ ]:
# CELL 6: Run Detailed Validation on Best Model Checkpoint
from pathlib import Path
from ultralytics import YOLO

print('=' * 70)
print('CELL 6: DETAILED VALIDATION ON BEST CHECKPOINT')
print('=' * 70)

best_model_path = Path('/kaggle/working/runs/yolo11s_baseline/weights/best.pt')
if not best_model_path.exists():
    raise FileNotFoundError(f'best.pt weights not found at {best_model_path}')

val_model = YOLO(str(best_model_path))

val_results = val_model.val(
    data='/kaggle/working/data_kaggle.yaml',
    split='val',
    imgsz=640,
    batch=32,
    device=0,
    project='/kaggle/working/runs',
    name='yolo11s_baseline_val',
    exist_ok=True,
    plots=True,
)

print('Validation complete.')


In [ ]:
# CELL 7: Display Important Metrics & Visual Diagnostics
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

print('=' * 70)
print('CELL 7: METRIC REPORT & DIAGNOSTICS')
print('=' * 70)

print(f'Overall Precision (P):     {val_results.box.mp:.4f}')
print(f'Overall Recall (R):        {val_results.box.mr:.4f}')
print(f'Overall mAP@50:            {val_results.box.map50:.4f}')
print(f'Overall mAP@50-95:         {val_results.box.map:.4f}')

print('\nPer-Class Metrics:')
print('-' * 70)
print(f"{'ID':<4} {'Class Name':<16} {'Precision':<12} {'Recall':<12} {'mAP@50':<12} {'mAP@50-95':<12}")
print('-' * 70)

for idx, name in EXPECTED_CLASSES.items():
    p_val = val_results.box.p[idx] if idx < len(val_results.box.p) else 0.0
    r_val = val_results.box.r[idx] if idx < len(val_results.box.r) else 0.0
    # Extract class-level mAP values robustly
    ap50_val = val_results.box.ap50[idx] if hasattr(val_results.box, 'ap50') and idx < len(val_results.box.ap50) else (val_results.box.maps[idx] if idx < len(val_results.box.maps) else 0.0)
    ap_val = val_results.box.maps[idx] if idx < len(val_results.box.maps) else 0.0
    print(f"{idx:<4} {name:<16} {p_val:<12.4f} {r_val:<12.4f} {ap50_val:<12.4f} {ap_val:<12.4f}")
print('-' * 70)

# Display training curves and confusion matrix
run_dir = Path('/kaggle/working/runs/yolo11s_baseline')
plots_to_show = [
    ('Training Loss & Metric Curves', run_dir / 'results.png'),
    ('Confusion Matrix', run_dir / 'confusion_matrix.png'),
    ('Normalized Confusion Matrix', run_dir / 'confusion_matrix_normalized.png'),
    ('Validation Predictions Sample', run_dir / 'val_batch0_pred.jpg'),
]

for title, img_path in plots_to_show:
    if img_path.exists():
        print(f'\n--- {title} ({img_path.name}) ---')
        display(Image(str(img_path)))


In [ ]:
# CELL 8: Package Essential Output Files for Download
from pathlib import Path
import zipfile

print('=' * 70)
print('CELL 8: PACKAGE ARTIFACTS FOR DOWNLOAD')
print('=' * 70)

run_dir = Path('/kaggle/working/runs/yolo11s_baseline')
val_dir = Path('/kaggle/working/runs/yolo11s_baseline_val')
zip_path = Path('/kaggle/working/yolo11s_baseline_artifacts.zip')

essential_files = [
    (run_dir / 'weights' / 'best.pt', 'weights/best.pt'),
    (run_dir / 'weights' / 'last.pt', 'weights/last.pt'),
    (run_dir / 'results.csv', 'results.csv'),
    (run_dir / 'args.yaml', 'args.yaml'),
    (run_dir / 'results.png', 'results.png'),
    (run_dir / 'confusion_matrix.png', 'confusion_matrix.png'),
    (run_dir / 'confusion_matrix_normalized.png', 'confusion_matrix_normalized.png'),
    (run_dir / 'labels.jpg', 'labels.jpg'),
]

with zipfile.ZipFile(zip_path, mode='w', compression=zipfile.ZIP_DEFLATED) as zf:
    for src, arcname in essential_files:
        if src.exists():
            zf.write(src, arcname=f'yolo11s_baseline/{arcname}')
            print(f'  Archived: {arcname:<30} ({src.stat().st_size / (1024**2):.2f} MB)')
        else:
            print(f'  Notice: {src.name} not found, skipping')

    for val_img in list(run_dir.glob('val_batch*_pred.jpg')) + list(val_dir.glob('val_batch*_pred.jpg')):
        zf.write(val_img, arcname=f'yolo11s_baseline/val_samples/{val_img.name}')
        print(f'  Archived sample: {val_img.name}')

zip_size_mb = zip_path.stat().st_size / (1024**2)
print('-' * 70)
print(f'Artifact archive created: {zip_path}')
print(f'Archive Size:             {zip_size_mb:.2f} MB')
print('\nDownload Instructions:')
print("1. Look at the right sidebar of the Kaggle notebook UI under 'Data' -> 'Output' -> '/kaggle/working'.")
print("2. Click the three dots next to 'yolo11s_baseline_artifacts.zip' and select 'Download'.")
print('3. Send or save best.pt and results.csv for the team evaluation.')
